# Homework 3 Work



Instead of including the full embeddings and the division of ratings.dat by timestamp in the DAG, I copied the embeddings from HW2 over to the new S3 bucket, and the below script divides ratings.dat, which I uploaded manually to S3. These are 1-time actions, so it felt pointless to put these in the MWAA flow. 

In [ ]:
# split ratings data into four partitions by timestamp.

# ratings.dat has timestamps in seconds since the epoch...
# need to divide based on these timestamps:

timestamp_ranges: list[list[str]] = [
    ["04/25/2000", "08/04/2000"],
    ["08/04/2000", "11/01/2000"],
    ["11/01/2000", "11/26/2000"],
    ["11/26/2000", "01/01/2027"]  # everything after 11/26/2000
]

# convert dates into seconds since the epoch (UTC midnight for each boundary)
from datetime import datetime, timezone

def to_epoch_seconds(date_str: str) -> int:
    dt = datetime.strptime(date_str, "%m/%d/%Y").replace(tzinfo=timezone.utc)
    return int(dt.timestamp())

timestamp_ranges_seconds: list[list[int]] = [
    [to_epoch_seconds(start), to_epoch_seconds(end)]
    for start, end in timestamp_ranges
]

timestamp_ranges_seconds

[[956620800, 965347200],
 [965347200, 973036800],
 [973036800, 975196800],
 [975196800, 1798761600]]

In [2]:
import pandas as pd

In [9]:
ratings = pd.read_csv("ml-1m/ratings.dat", sep="::", engine="python",
                    names=["user_id","movie_id","rating","timestamp"])

ratings.dtypes

user_id      int64
movie_id     int64
rating       int64
timestamp    int64
dtype: object

In [11]:
# split the ratings data by these timestamps

rating_date_partitions: dict[str, pd.DataFrame] = {}

for i, (start_ts, end_ts) in enumerate(timestamp_ranges_seconds):
    ratings_chunk = ratings[(ratings["timestamp"] >= start_ts) & (ratings["timestamp"] < end_ts)] 
    rating_date_partitions[f"ratings_{start_ts}-{end_ts}"] = (ratings_chunk)

for key, chunk in rating_date_partitions.items():
    chunk.info()
    print()
    path = f"ml-1m/{key}"
    chunk.to_csv(path)
    print(f"wrote csv to {path}")
    print()


<class 'pandas.core.frame.DataFrame'>
Index: 265435 entries, 692235 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    265435 non-null  int64
 1   movie_id   265435 non-null  int64
 2   rating     265435 non-null  int64
 3   timestamp  265435 non-null  int64
dtypes: int64(4)
memory usage: 10.1 MB

wrote csv to ml-1m/ratings_956620800-965347200

<class 'pandas.core.frame.DataFrame'>
Index: 235042 entries, 451428 to 994109
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    235042 non-null  int64
 1   movie_id   235042 non-null  int64
 2   rating     235042 non-null  int64
 3   timestamp  235042 non-null  int64
dtypes: int64(4)
memory usage: 9.0 MB

wrote csv to ml-1m/ratings_965347200-973036800

<class 'pandas.core.frame.DataFrame'>
Index: 245465 entries, 143479 to 993819
Data columns (total 4 columns):
 #   Column     Non-Null C

In [ ]:
from pathlib import Path
import os
import urllib.request
import zipfile

import boto3
from boto3.s3.transfer import S3UploadFailedError
from botocore.exceptions import ClientError
from dotenv import load_dotenv

Some utility functions adapted from HW2

```
You will run 4 iterations total. At each iteration:

Load the latest observation partition (the newly “arrived” data).
Handle newly added users (users that did not appear in earlier partitions).
Sample a random 30% subset of available users to compute user embeddings.
```

In [3]:
def read_ml1m(ml1m_folder: str="ml-1m") -> tuple[pd.DataFrame]:
    ratings = pd.read_csv(f"{ml1m_folder}/ratings.dat", sep="::", engine="python",
                      names=["user_id","movie_id","rating","timestamp"])
    movies  = pd.read_csv(f"{ml1m_folder}/movies.dat",  sep="::", engine="python",
                      names=["movie_id","title","genres"], encoding="latin-1")
    users   = pd.read_csv(f"{ml1m_folder}/users.dat", sep="::", engine="python",
                      names=["user_id", "gender", "age", "occupation", "zip"])
    return ratings,movies,users

ratings, movies, users = read_ml1m()

In [ ]:
def sample_users(users_df: pd.DataFrame, sample_pct: float) -> pd.DataFrame:
    """
    given a dataframe of users from the ml-1m dataset, returns a randomly sampled
    dataframe of `sample_pct`% of the users from `users_df`.
    """
    if not 0 < sample_pct <= 100:
        raise ValueError("sample_pct must a number >0 and <=100, got ", sample_pct)
    sample_frac = sample_pct / 100.0
    return users_df.sample(frac=sample_frac, random_state=42)

# I made this function so I can keep my sample consistent when generating embeddings, to
# be able to reuse my work for later tasks.
def sample_or_load_users(movie_lens_dir: str, sample_pct: float, users_df: pd.DataFrame = users) -> pd.DataFrame:
    """
    Attempts to load a sampled users file from ml-1m/users_sample_{sample_pct}.csv.
    If it does not exist, samples users_df and saves it to that path.
    """
    sample_file =  Path(movie_lens_dir) / f"users_sample_{int(sample_pct)}.csv"
    if sample_file.exists():
        print(f"{sample_file} already exists, loading df from csv")
        return pd.read_csv(sample_file)

    sampled_df = sample_users(users_df, sample_pct)
    sampled_df.to_csv(sample_file, index=False)
    return sampled_df

# global dataframe of all the users (user ratings) we have processed so far

# function to compute embeddings for a sample of users.

when building movie embeddings, try to run in batches of 128

In [3]:
from transformers import AutoTokenizer, AutoModel

In [ ]:
def get_bert_pretrained():
    DEVICE = "cpu"
    MODEL_NAME = "distilbert-base-uncased" # for illustrations, 66M model
    print(DEVICE, " | ", MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
    return tokenizer, encoder

tokenizer, encoder = get_bert_pretrained()
encoder.eval()

In [ ]:
def create_s3_client():
    """
    Initialize and return an S3 client given the following parameters
    in the current directory's .env:
    ```
    aws_access_key_id
    aws_secret_access_key
    aws_session_token
    region_name
    ```
    """
    
    load_dotenv('.env', override=True)

    session_kwargs = {
        "aws_access_key_id": os.getenv("AWS_ACCESS_KEY_ID"),
        "aws_secret_access_key": os.getenv("AWS_SECRET_ACCESS_KEY"),
        "aws_session_token": os.getenv("AWS_SESSION_TOKEN"),
        "region_name": os.getenv("AWS_REGION"),
    }

    session_kwargs = {k: v for k, v in session_kwargs.items() if v}

    aws_session = boto3.Session(**session_kwargs)
    s3_client = aws_session.client("s3")
    sts = aws_session.client("sts")
    
    return s3_client, sts

In [ ]:
def _get_bucket_name(arn_or_name: str) -> str:
    """Accept either a plain bucket name or an S3 bucket ARN and
    return the bucket name portion that the boto3 S3 client expects."""
    prefix = "arn:aws:s3:::"
    if arn_or_name.startswith(prefix):
        return arn_or_name[len(prefix) :]
    return arn_or_name

# Helper to upload any artifact to the configured S3 bucket
def try_upload_artifact(s3_client, bucket_name: str, local_path: Path, s3_key: str, override: bool=False):
    """Upload a local file to S3, skipping if the key already exists in the bucket.
    If `override` = `true`, upload anyways."""
    bucket = _get_bucket_name(bucket_name)
    if not local_path.exists():
        raise FileNotFoundError(f"Missing artifact: {local_path}")
    
    # Check if the S3 key already exists
    try:
        s3_client.head_object(Bucket=bucket, Key=s3_key)
        if override:
            print(f"S3 key '{s3_key}' already exists in bucket '{bucket}', attempting upload anyways.")
        else:
            print(f"S3 key '{s3_key}' already exists in bucket '{bucket}', skipping upload.")
            return
    except ClientError as e:
        # If a 404 error, the object does not exist, so proceed with upload
        if e.response['Error']['Code'] == '404':
            pass
        else:
            # Some other error occurred
            raise
    
    print(f"Uploading {local_path} -> s3://{bucket}/{s3_key}")
    s3_client.upload_file(str(local_path), bucket, s3_key)

In [ ]:
# cold user: not in the sampled user set
def get_cold_user(users, users_sample):
    all_user_ids = set(users["user_id"])
    user_subset = set(users_sample["user_id"].tolist())
    cold_user_candidates = all_user_ids - user_subset
    cold_user = sorted(cold_user_candidates)[0]  # pick first user for consistency
    print(f"Cold user: {cold_user}")
    return cold_user

# identify a top user: in the top 5% by number of interactions
def get_top_user(sample_ratings): 
    user_interaction_counts = sample_ratings.groupby("user_id").size().reset_index(name="interaction_count")
    top_5_percent_threshold = user_interaction_counts["interaction_count"].quantile(0.95)
    top_users = user_interaction_counts[user_interaction_counts["interaction_count"] >= top_5_percent_threshold]
    top_user = top_users.sample(1, random_state=42)["user_id"].iloc[0]
    print(f"Top user: {top_user} with {user_interaction_counts[user_interaction_counts['user_id']==top_user]['interaction_count'].iloc[0]} interactions")
    return top_user

In [ ]:
# Build positive interactions restricted to the sampled 30% of users like we did in lab
def build_positive_ratings(ratings, users_sample):
    ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
    user_subset = set(users_sample["user_id"].tolist())
    sample_ratings = ratings[ratings["user_id"].isin(user_subset)]
    pos = sample_ratings[sample_ratings["rating"] >= 4].copy()
    pos["value"] = 1.0
    pos = pos.sort_values(["user_id", "timestamp"])

    events = pos[["user_id", "movie_id", "timestamp"]].rename(columns={"timestamp": "ts"})
    print(
        f"Sample users: {len(user_subset)} | positive interactions: {len(pos)} | unique movies: {pos['movie_id'].nunique()}"
    )

    return events, sample_ratings

In [ ]:
import numpy as np
import torch
import faiss

In [ ]:
# Helpers for task 3 recommendations (sampled 30% users)

def _encode_single_text(text: str, max_len: int = 128) -> np.ndarray:
    DEVICE = "cpu"
    if not text:
        return None
    with torch.no_grad():
        batch = tokenizer(
            [text], padding=True, truncation=True, max_length=max_len, return_tensors="pt"
        ).to(DEVICE)
        out = encoder(**batch).last_hidden_state[:, 0, :]
        emb = F.normalize(out, p=2, dim=1).cpu().numpy().astype("float32")
        return emb


def build_user_history(user_id: int, events_df: pd.DataFrame, movies_df: pd.DataFrame, n: int = 10):
    user_events = events_df[events_df["user_id"] == user_id].sort_values("ts")
    last_ts = user_events["ts"].max() if not user_events.empty else None
    seen = set(user_events["movie_id"].tolist())
    texts = []
    if not user_events.empty:
        recent = user_events.tail(n)["movie_id"].tolist()
        texts = movies_df.set_index("movie_id").loc[recent, "text"].fillna("").tolist()
    return texts, seen, last_ts


# taken and modified from lab 4, used by task-specific recommender functions
def recommend(
    user_id: int,
    movies_df: pd.DataFrame,
    events_df: pd.DataFrame,
    idx: faiss.IndexFlatIP,
    k: int = 5,
    history_n: int = 10, # most recent user ratings
    fallback_ratings: pd.DataFrame | None = None,
    popular_min_rating: int = 4,
):
    """Shared recommender: uses history when present, otherwise popularity fallback."""
    texts, seen, last_ts = build_user_history(user_id, events_df, movies_df, n=history_n)

    if texts:
        user_text = " ".join(texts)
        u = _encode_single_text(user_text)
        scores, idxs = idx.search(u, k + len(seen) + 20)
        recs = []
        for j in idxs[0]:
            mid = int(movies_df.iloc[j]["movie_id"])
            if mid in seen:
                continue
            recs.append(mid)
            if len(recs) == k:
                break
    else:
        recs = []
        if fallback_ratings is not None:
            top_popular = (
                fallback_ratings[fallback_ratings["rating"] >= popular_min_rating]
                .groupby("movie_id")["rating"]
                .size()
                .sort_values(ascending=False)
                .head(k)
                .index.tolist()
            )
            recs = top_popular
    return {"user_id": user_id, "last_ts": last_ts, "recs": recs, "seen": list(seen)}


def recommend_for_user(user_id: int, sample_ratings: pd.DataFrame, k: int = 5):
    # wrapper for sampled users/events/index
    return recommend(
        user_id=user_id,
        movies_df=movies,
        events_df=events,
        idx=index,
        k=k,
        history_n=10,
        fallback_ratings=sample_ratings,
    )


def summarize_user(user_id: int, user_type: str, sample_ratings: pd.DataFrame):
    rec_info = recommend_for_user(user_id, sample_ratings, k=5)
    row = users[users["user_id"] == user_id]
    summary = {
        "User_Type": user_type,
        "User_ID": user_id,
        "Last_Interaction_Time": rec_info["last_ts"],
        "Interactions": len(rec_info["seen"]),
        "Recs": rec_info["recs"],
    }
    if not row.empty:
        summary.update({
            "Gender": row.iloc[0]["gender"],
            "Age": row.iloc[0]["age"],
            "Occupation": row.iloc[0]["occupation"],
            "Zip": row.iloc[0]["zip"],
        })
    return summary


def orchestrate_task3():
    cold_user = get_cold_user(users, users_sample)
    top_user = get_top_user(sample_ratings)
    records = [summarize_user(cold_user, "cold", sample_ratings), summarize_user(top_user, "top", sample_ratings)]
    df = pd.DataFrame(records)
    out_path = MOVIELENS_DIR / "task3_recommendations.csv"
    df.to_csv(out_path, index=False)
    try_upload_artifact(out_path, "ml-1m/task3_recommendations.csv")
    return df

# Generative AI Disclosure

```
Please convert the timestamps strings into seconds since the epoch that are consistent with the schema for #file:ratings.dat defined in #file:README
```